<a href="https://colab.research.google.com/github/braim/nids-tta/blob/kan-ton-1000-splinefalse/colab/9-vis/NIDS_CTTA_VIS_8_0_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NIDS — AE+Classifier with Few-Shot Layer-Selective CTTA

*   KAN
*   TON as Source
*   1000k
*   Spline-Only = False


## Design

**Pre-training:** supervised CE + reconstruction MSE on fully labelled source.

**CTTA strategy — Layer-Selective Updates:**
Three parameter groups are updated during CTTA:
1. **Norm layers** (LayerNorm, GroupNorm) — adapt feature normalisation statistics
2. **Classifier head** — directly moves the decision boundary toward target patterns
3. **Last encoder layer** — adapts the final feature representation

The early encoder layers (source feature extractors) remain frozen.
This gives the CE loss a direct path to update the decision boundary
while protecting source representations from catastrophic forgetting.

**Three loss signals per batch (all requiring only 1% labelled pool):**
1. Supervised CE on labelled pool — moves decision boundary to target domain
2. Entropy minimisation on stream — increases prediction confidence
3. Reconstruction on pool — keeps representations grounded

## Architectures
| `ARCH` | Description |
|---|---|

| `'kan'` | KAN encoder/decoder + linear classifier |
| `'cnn'` | CNN encoder/decoder + linear classifier |

## Hyperparameters
| Parameter | Description |
|---|---|
| `RECON_W` | Reconstruction weight during pre-training |
| `FEW_SHOT_RATIO` | Fraction of target data used as labelled pool |
| `FEW_SHOT_W` | Supervised CE weight during CTTA |
| `TTA_LR` | Learning rate for layer-selective updates |
| `TTA_STEPS` | Gradient steps per batch |
| `ENTROPY_W` | Entropy minimisation weight |
| `RECON_W_TTA` | Reconstruction weight during CTTA |

In [1]:
!pip install -q git+https://github.com/Blealtan/efficient-kan.git
!pip install -q polars kagglehub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Imports & Configuration

In [2]:
import os, gc, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import polars as pl
import kagglehub
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
from efficient_kan import KAN
from datetime import datetime,timezone
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
############################ TO CHANGE ACROSS RUNS #############################
# ── Architecture ──────────────────────────────────────────────────────────────
ARCH            = 'kan'      # 'kan' | 'cnn'
SPLINE_TRUE      = False

# ── Datasets ──────────────────────────────────────────────────────────────────
SOURCE_DATASET  = 'seyhed/nf-ton-iot-v3'
TARGET1_DATASET = 'seyhed/nf-unsw-nb15-v3'
TARGET2_DATASET = 'seyhed/nf-cicids2018-v3'

SOURCE_NAME     = 'ToN-IoT'
TARGET1_NAME    = 'UNSW-NB15'
TARGET2_NAME    = 'CICIDS2018'

# ── Data ──────────────────────────────────────────────────────────────────────
SAMPLE_N        = 1000_000
################################################################################
BATCH_SIZE      = 256
TTA_BATCH_SIZE  = 512        # larger batches = more stable gradient estimates

# ── Model ─────────────────────────────────────────────────────────────────────
LATENT_DIM      = 32

# ── Pre-training ──────────────────────────────────────────────────────────────
PRETRAIN_EPOCHS = 20
PRETRAIN_LR     = 1e-3
WEIGHT_DECAY    = 1e-4
RECON_W         = 0.5        # reconstruction regularises encoder for transfer

# ── CTTA ──────────────────────────────────────────────────────────────────────
FEW_SHOT_RATIO  = 0.001       # fraction of target used as benign pool
FEW_SHOT_W      = 1.0        # supervised CE weight during CTTA
TTA_LR          = 1e-2       # higher lr is fine — only norm params updated
TTA_STEPS       = 1
ENTROPY_W       = 1.0        # entropy minimisation weight
RECON_W_TTA     = 0.5        # reconstruction on benign pool weight
# from google.colab import drive
# drive.mount('/content/drive')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device={device} | Arch={ARCH} | Source={SOURCE_NAME} | Size={SAMPLE_N} | Last Layer={SPLINE_TRUE}')
print(f"Last exec: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%SZ')}")

Device=cpu | Arch=kan | Source=ToN-IoT | Size=1000000 | Last Layer=False
Last exec: 2026-05-26 02:31:00Z


## 2. Data Loading & Preprocessing

In [3]:
def engineer_features(df: pl.DataFrame) -> pl.DataFrame:
    """Derive flow-level features and drop identifier/label columns."""
    if 'FLOW_END_MILLISECONDS' in df.columns and 'FLOW_START_MILLISECONDS' in df.columns:
        df = df.with_columns(
            (pl.col('FLOW_END_MILLISECONDS') - pl.col('FLOW_START_MILLISECONDS')).alias('FLOW_DURATION')
        )
    else:
        df = df.with_columns(pl.lit(0).alias('FLOW_DURATION'))
    if 'IN_BYTES' in df.columns and 'IN_PKTS' in df.columns:
        df = df.with_columns(
            (pl.col('IN_BYTES') / (pl.col('IN_PKTS') + 1e-5)).alias('BYTES_PER_PKT')
        )
    log_cols = ['IN_BYTES', 'IN_PKTS', 'FLOW_DURATION', 'SRC_TO_DST_IAT_MAX', 'DST_TO_SRC_IAT_MAX']
    existing = [c for c in log_cols if c in df.columns]
    if existing:
        df = df.with_columns([pl.col(c).log1p() for c in existing])
    drop_cols = [
        'FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS',
        'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT',
        'Label', 'Attack', 'label', 'attack', 'Date',
    ]
    df = df.drop([c for c in drop_cols if c in df.columns])
    return df


def load_dataset(dataset_name: str, sample_n: int = None):
    """Download dataset and return (X, y). Uses random sampling."""
    print(f'[Data] Loading {dataset_name} ...')
    path = kagglehub.dataset_download(dataset_name)
    csv_files = [
        os.path.join(root, f)
        for root, _, files in os.walk(path)
        for f in files if f.endswith('.csv')
    ]
    df = pl.scan_csv(csv_files[0]).collect(engine='streaming')
    if sample_n and sample_n < df.height:
        df = df.sample(n=sample_n, seed=SEED)
    label_col = next((c for c in df.columns if c.lower() == 'label'), None)
    y = df[label_col].to_numpy().astype(np.int64) if label_col else np.zeros(df.height, dtype=np.int64)
    df = engineer_features(df)
    X  = df.to_numpy().astype(np.float32)
    X  = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    print(f'   -> Shape: {X.shape} | Attack rate: {np.mean(y):.2%}')
    return X, y


def make_source_loaders(X, y):
    """
    Stratified 80/20 split. Scaler fitted on full training split.
    Training loader contains all labelled samples (benign + attack).
    """
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    scaler = MinMaxScaler(feature_range=(-1, 1)).fit(X_tr)
    X_tr   = np.clip(scaler.transform(X_tr).astype(np.float32), -1, 1)
    X_te   = np.clip(scaler.transform(X_te).astype(np.float32), -1, 1)
    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    test_ds  = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
    loader_tr = DataLoader(train_ds, batch_size=BATCH_SIZE,     shuffle=True)
    loader_te = DataLoader(test_ds,  batch_size=TTA_BATCH_SIZE, shuffle=False)
    return loader_tr, loader_te, scaler


def make_target_loaders(X, y, external_scaler=None):
    """
    Fit a fresh MinMaxScaler on the full target dataset (no label leakage).

    Stratified split into:
      pool_loader   : FEW_SHOT_RATIO of data, labelled (both classes preserved)
      stream_loader : remaining data, labels kept for evaluation only

    The pool is used for supervised CE anchoring during CTTA.
    Real-world justification: the pool represents a brief initial analyst
    review period at deployment — a realistic assumption.
    """


    # Stratified split — both classes represented in pool
    X_pool, X_stream, y_pool, y_stream = train_test_split(
    X, y,
    test_size=(1 - FEW_SHOT_RATIO),
    random_state=SEED,
    stratify=y,
    )
    scaler   = external_scaler if external_scaler is not None else MinMaxScaler(feature_range=(-1, 1)).fit(X_pool)

    X_pool   = np.clip(scaler.transform(X_pool).astype(np.float32), -1, 1)
    X_stream = np.clip(scaler.transform(X_stream).astype(np.float32), -1, 1)

    pool_ds   = TensorDataset(torch.from_numpy(X_pool),   torch.from_numpy(y_pool))
    stream_ds = TensorDataset(torch.from_numpy(X_stream), torch.from_numpy(y_stream))

    pool_loader   = DataLoader(pool_ds,   batch_size=BATCH_SIZE,     shuffle=True)
    stream_loader = DataLoader(stream_ds, batch_size=TTA_BATCH_SIZE, shuffle=False)

    print(f'   -> Pool: {len(y_pool)} samples '
          f'(attack rate: {np.mean(y_pool):.2%}) | '
          f'Stream: {len(y_stream)} '
          f'(attack rate: {np.mean(y_stream):.2%})')
    return pool_loader, stream_loader


## 3. Model Architectures

In [4]:
class KanAEClassifier(nn.Module):
    """
    Shared KAN encoder → classifier head + decoder head.

    Encoder:    input_dim -> 64 -> latent_dim  (KAN)
    Classifier: latent_dim -> 2               (Linear)
    Decoder:    latent_dim -> 64 -> input_dim  (KAN)

    forward() returns (logits, recon, z)
    """
    def __init__(self, input_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder    = KAN([input_dim, 64, latent_dim], grid_range=[-1, 1])
        self.ln         = nn.LayerNorm(latent_dim)
        self.classifier = nn.Linear(latent_dim, 2)
        self.decoder    = KAN([latent_dim, 64, input_dim], grid_range=[-1, 1])

    def forward(self, x):
        z = self.ln(self.encoder(x))
        return self.classifier(z), self.decoder(z), z



class TabTransformerAEClassifier(nn.Module):
    """
    Shared TabTransformer encoder → classifier head + decoder head.

    Encoder:    input_dim -> 64 -> latent_dim  (Attention + Linear)
    Classifier: latent_dim -> 2               (Linear)
    Decoder:    latent_dim -> 64 -> input_dim  (Linear)

    forward() returns (logits, recon, z)
    """
    def __init__(self, input_dim: int, latent_dim: int = 32,
                n_heads: int = 4, d_token: int = 16):
        super().__init__()
        self.input_dim = input_dim
        self.d_token   = d_token
        # Each scalar feature -> d_token vector (shared linear + per-feature bias)
        self.feature_proj = nn.Linear(1, d_token, bias=False)
        self.feature_bias = nn.Parameter(torch.randn(input_dim, d_token) * 0.02)
        self.ln1 = nn.LayerNorm(d_token)
        self.attn = nn.MultiheadAttention(d_token, n_heads, batch_first=True)
        self.ln2 = nn.LayerNorm(d_token)
        self.ffn = nn.Sequential(
            nn.Linear(d_token, d_token * 4), nn.GELU(),
            nn.Linear(d_token * 4, d_token),
        )
        self.encoder_out = nn.Linear(d_token, latent_dim)
        self.ln          = nn.LayerNorm(latent_dim)
        self.classifier  = nn.Linear(latent_dim, 2)
        self.decoder     = nn.Sequential(
            nn.Linear(latent_dim, d_token),
            nn.LayerNorm(d_token), nn.GELU(),
            nn.Linear(d_token, input_dim),
        )

    def forward(self, x):
        # x: (B, D) -> tokens: (B, D, d_token)
        tokens = self.feature_proj(x.unsqueeze(-1)) + self.feature_bias.unsqueeze(0)
        tokens = self.ln1(tokens)
        attn_out, _ = self.attn(tokens, tokens, tokens)
        tokens = tokens + attn_out
        tokens = self.ln2(tokens)
        tokens = tokens + self.ffn(tokens)
        pooled = tokens.mean(dim=1)
        z = self.ln(self.encoder_out(pooled))
        return self.classifier(z), self.decoder(z), z

class CnnAEClassifier(nn.Module):
    """
    Shared CNN encoder → classifier head + decoder head.

    Each feature becomes its own channel (seq len=1). Pointwise Conv1d.
    GroupNorm(1, C) works with seq len=1 and is robust to variable attack rates.

    Encoder:    input_dim -> 64 -> latent_dim  (Conv1d)
    Classifier: latent_dim -> 2               (Linear)
    Decoder:    latent_dim -> 64 -> input_dim  (ConvTranspose1d)

    forward() returns (logits, recon, z)
    """
    def __init__(self, input_dim: int, latent_dim: int = 32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(input_dim, 64, kernel_size=1),
            nn.GroupNorm(1, 64), nn.GELU(),
            nn.Conv1d(64, latent_dim, kernel_size=1),
        )
        self.ln         = nn.LayerNorm(latent_dim)
        self.classifier = nn.Linear(latent_dim, 2)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(latent_dim, 64, kernel_size=1),
            nn.GroupNorm(1, 64), nn.GELU(),
            nn.ConvTranspose1d(64, input_dim, kernel_size=1),
        )

    def forward(self, x):
        z     = self.ln(self.encoder(x.unsqueeze(-1)).squeeze(-1))
        recon = self.decoder(z.unsqueeze(-1)).squeeze(-1)
        return self.classifier(z), recon, z


def build_model(arch: str, input_dim: int) -> nn.Module:
    if arch == 'kan':
        return KanAEClassifier(input_dim, LATENT_DIM)
    elif arch == 'cnn':
        return CnnAEClassifier(input_dim, LATENT_DIM)
    elif arch == 'tab':
        return TabTransformerAEClassifier(input_dim, LATENT_DIM)
    else:
        raise ValueError(f"Unknown ARCH={arch!r}. Choose 'kan', 'cnn', or 'tab'")


def get_trainable_params(model: nn.Module):
    """
    Return parameters for layer-selective CTTA updates.

    Updated:
      - All LayerNorm / GroupNorm parameters (normalisation statistics)
      - Classifier head (decision boundary can shift toward target)
      - Last encoder layer (final feature representation can adapt)

    Frozen:
      - All early encoder layers (source feature extractors)
      - Decoder (not needed for classification)

    This gives CE a direct path to move the decision boundary while
    protecting source representations from catastrophic forgetting.
    """
    params = []
    seen   = set()

    def add(p):
        if id(p) not in seen:
            seen.add(id(p))
            params.append(p)

    # 1. Norm layers (encoder + classifier only, not decoder)
    for name, module in model.named_modules():
        if 'decoder' in name:
            continue
        if isinstance(module, (nn.LayerNorm, nn.GroupNorm)):
            for p in module.parameters():
                add(p)

    # 2. Classifier head
    for p in model.classifier.parameters():
        add(p)

    # 3. Last encoder layer
    if isinstance(model, KanAEClassifier):
        # KAN — last KANLayer
        if hasattr(model.encoder, 'layers') and len(model.encoder.layers) > 0:
            for p in model.encoder.layers[-1].parameters():
                add(p)
    elif isinstance(model, CnnAEClassifier):
        # CNN — last Conv1d in sequential
        last_layer = None
        for m in model.encoder.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                last_layer = m
        if last_layer is not None:
            for p in last_layer.parameters():
                add(p)
    elif isinstance(model, TabTransformerAEClassifier):
        # TabTransformer — encoder_out projection layer
        for p in model.encoder_out.parameters():
            add(p)

    return params


## 4. Training & Evaluation

In [5]:
class KAN_Retention:
    """
    KAN-specific Elastic Weight Consolidation (EWC).
    Calculates the Fisher Information for specific KAN parameters (e.g., splines)
    to penalize drastic changes to important learned shapes during CTTA.
    """
    def __init__(self, model, dataloader, device, params_to_protect):
        self.model = model
        self.dataloader = dataloader
        self.device = device
        self.params_to_protect = params_to_protect

        # 1. Store the optimal source parameters (theta_source)
        self.optpar_dict = {}
        for p in self.params_to_protect:
            self.optpar_dict[id(p)] = p.data.clone().detach()

        # 2. Compute Fisher Information Matrix (diagonal approximation)
        print('[KAN Retention] Computing Fisher Information on Source Data...')
        self.fisher_dict = self._compute_fisher()
        print('[KAN Retention] Initialization Complete.')

    def _compute_fisher(self):
        fisher_dict = {id(p): torch.zeros_like(p.data) for p in self.params_to_protect}

        # Ensure we can compute gradients for these parameters
        for p in self.params_to_protect:
            p.requires_grad_(True)

        self.model.eval() # Eval mode for stability
        ce_crit = nn.CrossEntropyLoss()

        num_samples = 0
        for x, y in self.dataloader:
            x, y = x.to(self.device), y.to(self.device)
            self.model.zero_grad()

            logits, _, _ = self.model(x)
            loss = ce_crit(logits, y)
            loss.backward()

            # Accumulate squared gradients (Fisher diagonal)
            for p in self.params_to_protect:
                if p.grad is not None:
                    fisher_dict[id(p)] += (p.grad.data ** 2) * x.size(0)

            num_samples += x.size(0)

        # Average over all samples
        for p in self.params_to_protect:
            fisher_dict[id(p)] /= num_samples

        return fisher_dict

    def penalty(self):
        """
        Computes the EWC penalty: sum( Fisher * (theta - theta_source)^2 )
        """
        loss = 0.0
        for p in self.params_to_protect:
            fisher = self.fisher_dict[id(p)]
            optpar = self.optpar_dict[id(p)]
            loss += (fisher * (p - optpar) ** 2).sum()
        return loss

In [6]:
def pretrain_source(model, loader, epochs, device):
    """
    Joint supervised + reconstruction pre-training.
    Loss = CrossEntropy(logits, y) + RECON_W * MSE(recon, x)
    ALL parameters updated — standard supervised training.
    """
    optimizer = optim.Adam(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
    ce_crit   = nn.CrossEntropyLoss()
    mse_crit  = nn.MSELoss()
    model.train()

    for epoch in range(epochs):
        total_loss, correct, total = 0.0, 0, 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits, recon, _ = model(x)
            loss = ce_crit(logits, y) + RECON_W * mse_crit(recon, x)
            if not (torch.isnan(loss) or torch.isinf(loss)):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += loss.item()
            correct    += (logits.argmax(1) == y).sum().item()
            total      += y.size(0)
        print(f'[Pretrain] Epoch {epoch+1}/{epochs} | '
              f'Loss: {total_loss/len(loader):.4f} | '
              f'Acc: {correct/total:.4f}')


def evaluate(model, loader, device, desc='Eval'):
    """Evaluate using classifier head — argmax of logits."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            logits, _, _ = model(x.to(device))
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.numpy())
    preds  = np.array(all_preds)
    labels = np.array(all_labels)
    f1  = f1_score(labels, preds, zero_division=0)
    acc = accuracy_score(labels, preds)
    print(f'[{desc}] F1: {f1:.4f} | Acc: {acc:.4f}')
    return f1

def get_spline_only_params(model):
    """
    Return the minimal-parameter trainable set for CTTA, per architecture.

    For KAN: ONLY the spline parameters of the last encoder layer
             (spline_weight + spline_scaler). base_weight is EXCLUDED —
             it's a linear residual path, not part of the spline.
    For CNN: parameters of the last Conv1d / Linear in the encoder
             (no splines exist in this arch).
    For TabTransformer: parameters of encoder_out, the final projection
             into the latent space (no splines in this arch).

    The kwarg name `spline_only` is kept for backward compatibility, but
    the semantics differ by architecture — for non-KAN it is really
    'last-layer-only'.
    """
    params = []

    if isinstance(model, KanAEClassifier):
        if not hasattr(model.encoder, 'layers') or len(model.encoder.layers) == 0:
            return []
        last = model.encoder.layers[-1]
        if hasattr(last, 'spline_weight') and isinstance(last.spline_weight, torch.nn.Parameter):
            params.append(last.spline_weight)
        if hasattr(last, 'spline_scaler') and isinstance(last.spline_scaler, torch.nn.Parameter):
            params.append(last.spline_scaler)

    elif isinstance(model, CnnAEClassifier):
        last_layer = None
        for m in model.encoder.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                last_layer = m
        if last_layer is not None:
            for p in last_layer.parameters():
                params.append(p)

    elif isinstance(model, TabTransformerAEClassifier):
        for p in model.encoder_out.parameters():
            params.append(p)

    return params


def run_ctta(model, stream_loader, pool_loader, device, spline_only=False):
    """
    Few-Shot Layer-Selective CTTA.

    FROZEN:  early encoder layers, decoder weights.
    UPDATED: LayerNorm/GroupNorm params, classifier head, last encoder layer.
             In spline_only mode: only last encoder layer spline weights.

    Per stream batch:
      1. Supervised CE on pool batch — anchors decision boundary to target
      2. Entropy minimisation on stream batch — increases prediction confidence
      3. Reconstruction on benign pool samples — prevents representation drift

    Returns (preds, labels, trajectory) on the stream.
    """
    if spline_only:
        train_params = get_spline_only_params(model)
        print(f'[CTTA] SPLINE-ONLY mode: updating only last encoder layer spline weights.')
    else:
        train_params = get_trainable_params(model)


    trainable_ids = {id(p) for p in train_params}
    for p in model.parameters():
        p.requires_grad = (id(p) in trainable_ids)

    print(f'[CTTA] Updating {len(train_params)} param tensors '
          f'({sum(p.numel() for p in train_params)} params). '
          f'All other weights frozen.')

    optimizer = optim.Adam(train_params, lr=TTA_LR)
    ce_crit   = nn.CrossEntropyLoss()
    model.train()

    # Pool cycles indefinitely
    pool_iter = iter(pool_loader)
    def next_pool():
        nonlocal pool_iter
        try:
            return next(pool_iter)
        except StopIteration:
            pool_iter = iter(pool_loader)
            return next(pool_iter)

    all_preds, all_labels = [], []
    trajectory = []
    batch_count = 0
    TRAJ_INTERVAL = 20

    for x_stream, y_stream in stream_loader:
        x_stream = x_stream.to(device)

        for _ in range(TTA_STEPS):
            optimizer.zero_grad()

            # ── 1. Supervised CE on pool batch ────────────────────────────
            x_pool, y_pool = next_pool()
            x_pool = x_pool.to(device)
            y_pool = y_pool.to(device)
            logits_pool, recon_pool, _ = model(x_pool)
            loss_ce    = ce_crit(logits_pool, y_pool)

            # ── 2. Entropy on stream batch ────────────────────────────────
            logits_s, _, _ = model(x_stream)
            probs    = F.softmax(logits_s, dim=1)
            loss_ent = -torch.sum(probs * torch.log(probs + 1e-8), dim=1).mean()

            # ── 3. Reconstruction on pool batch ───────────────────────────
            benign_mask = (y_pool == 0)
            if benign_mask.any():
                loss_recon = F.mse_loss(recon_pool[benign_mask], x_pool[benign_mask])
            else:
                loss_recon = torch.tensor(0.0, device=device)


            loss = (FEW_SHOT_W  * loss_ce   +
                    ENTROPY_W   * loss_ent  +
                    RECON_W_TTA * loss_recon)



            if not (torch.isnan(loss) or torch.isinf(loss)):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(train_params, max_norm=1.0)
                optimizer.step()

        # Final inference
        with torch.no_grad():
            model.eval()
            logits_f, _, _ = model(x_stream)
            preds = logits_f.argmax(1).cpu().numpy()
            model.train()

        all_preds.extend(preds)
        all_labels.extend(y_stream.numpy())

        batch_count += 1
        if batch_count % TRAJ_INTERVAL == 0:
            f1_so_far = f1_score(all_labels, all_preds, zero_division=0)
            trajectory.append((batch_count, f1_so_far))

    print('[CTTA] Stream complete.')
    return np.array(all_preds), np.array(all_labels), trajectory


## 5. Load Datasets

In [7]:
# ── Source: ───────────────────────────────────────────────────────────
X_src, y_src = load_dataset(SOURCE_DATASET, sample_n=SAMPLE_N)
input_dim = X_src.shape[1]
loader_src_train, loader_src_test, source_scaler = make_source_loaders(X_src, y_src)
del X_src, y_src; gc.collect()

# ── Target 1: ──────────────────────────────────────────────────────
X_tgt1, y_tgt1 = load_dataset(TARGET1_DATASET, sample_n=SAMPLE_N)
pool_tgt1, stream_tgt1 = make_target_loaders(X_tgt1, y_tgt1)
# Also build loaders using source scaler for the scaler experiment
pool_tgt1_src_sc, stream_tgt1_src_sc = make_target_loaders(X_tgt1, y_tgt1, external_scaler=source_scaler)
del X_tgt1, y_tgt1; gc.collect()

# ── Target 2: ─────────────────────────────────────────────────────
X_tgt2, y_tgt2 = load_dataset(TARGET2_DATASET, sample_n=SAMPLE_N)
pool_tgt2, stream_tgt2 = make_target_loaders(X_tgt2, y_tgt2)
pool_tgt2_src_sc, stream_tgt2_src_sc = make_target_loaders(X_tgt2, y_tgt2, external_scaler=source_scaler)
del X_tgt2, y_tgt2; gc.collect()

print(f'\n[System] All datasets loaded. Input dim: {input_dim}')

[Data] Loading seyhed/nf-ton-iot-v3 ...
Using Colab cache for faster access to the 'nf-ton-iot-v3' dataset.
   -> Shape: (1000000, 49) | Attack rate: 38.97%
[Data] Loading seyhed/nf-unsw-nb15-v3 ...
Using Colab cache for faster access to the 'nf-unsw-nb15-v3' dataset.
   -> Shape: (1000000, 49) | Attack rate: 5.44%
   -> Pool: 1000 samples (attack rate: 5.40%) | Stream: 999000 (attack rate: 5.44%)
   -> Pool: 1000 samples (attack rate: 5.40%) | Stream: 999000 (attack rate: 5.44%)
[Data] Loading seyhed/nf-cicids2018-v3 ...
Using Colab cache for faster access to the 'nf-cicids2018-v3' dataset.
   -> Shape: (1000000, 49) | Attack rate: 12.98%
   -> Pool: 1000 samples (attack rate: 13.00%) | Stream: 999000 (attack rate: 12.98%)
   -> Pool: 1000 samples (attack rate: 13.00%) | Stream: 999000 (attack rate: 12.98%)

[System] All datasets loaded. Input dim: 49


## 6. Phase 1 — Source Pre-training

In [8]:
# checkpoint = torch.load(f'/content/drive/MyDrive/pretrained_{ARCH}_toniot.pt')
# model = build_model(checkpoint['arch'], checkpoint['input_dim']).to(device)
# model.load_state_dict(checkpoint['model_state_dict'])
# print('Model loaded from Drive')


print('=' * 60)
print(f'PHASE 1: SOURCE PRE-TRAINING ({SOURCE_NAME}) | {ARCH.upper()}')
print('=' * 60)

model = build_model(ARCH, input_dim).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_params = get_trainable_params(model)
print(f'[Model] {ARCH} | input_dim={input_dim} | latent_dim={LATENT_DIM} | '
      f'total params={n_params:,} | trainable params={sum(p.numel() for p in trainable_params)}\n')

pretrain_source(model, loader_src_train, epochs=PRETRAIN_EPOCHS, device=device)

PHASE 1: SOURCE PRE-TRAINING (ToN-IoT) | KAN
[Model] kan | input_dim=49 | latent_dim=32 | total params=103,810 | trainable params=20610

[Pretrain] Epoch 1/20 | Loss: 0.2523 | Acc: 0.9003
[Pretrain] Epoch 2/20 | Loss: 0.2134 | Acc: 0.9135
[Pretrain] Epoch 3/20 | Loss: 0.2089 | Acc: 0.9166
[Pretrain] Epoch 4/20 | Loss: 0.2055 | Acc: 0.9186
[Pretrain] Epoch 5/20 | Loss: 0.2033 | Acc: 0.9202
[Pretrain] Epoch 6/20 | Loss: 0.2006 | Acc: 0.9227
[Pretrain] Epoch 7/20 | Loss: 0.1994 | Acc: 0.9233
[Pretrain] Epoch 8/20 | Loss: 0.1973 | Acc: 0.9252
[Pretrain] Epoch 9/20 | Loss: 0.1950 | Acc: 0.9271
[Pretrain] Epoch 10/20 | Loss: 0.1914 | Acc: 0.9294
[Pretrain] Epoch 11/20 | Loss: 0.1863 | Acc: 0.9319
[Pretrain] Epoch 12/20 | Loss: 0.1827 | Acc: 0.9335
[Pretrain] Epoch 13/20 | Loss: 0.1810 | Acc: 0.9350
[Pretrain] Epoch 14/20 | Loss: 0.1807 | Acc: 0.9356
[Pretrain] Epoch 15/20 | Loss: 0.1794 | Acc: 0.9364
[Pretrain] Epoch 16/20 | Loss: 0.1789 | Acc: 0.9365
[Pretrain] Epoch 17/20 | Loss: 0.1777 | 

## 7. Diagnostic — Post-Pretraining

In [10]:
# Source F1 should be >0.85 before CTTA.
# Zero-shot gives the baseline CTTA should improve from.
model_state = {k: v.clone() for k, v in model.state_dict().items()}
print('[Diagnostic] Source test performance:')
evaluate(model, loader_src_test, device, desc=f'Source ({SOURCE_NAME}) [post-pretrain]')

print('\n[Diagnostic] Zero-shot on targets (no adaptation yet):')
evaluate(model, stream_tgt1, device, desc=f'Target1 ({TARGET1_NAME})  [zero-shot]')
evaluate(model, stream_tgt2, device, desc=f'Target2 ({TARGET2_NAME}) [zero-shot]')

print('\n[Diagnostic] Parameters that will be updated during CTTA:')
ctta_params = get_trainable_params(model)
total_ctta = sum(p.numel() for p in ctta_params)
spline_params = get_spline_only_params(model)
total_spline = sum(p.numel() for p in spline_params)
total_all = sum(p.numel() for p in model.parameters())
print(f'  Full CTTA:   {total_ctta:,} / {total_all:,} params ({total_ctta/total_all:.2%})')
print(f'  Spline-only: {total_spline:,} / {total_all:,} params ({total_spline/total_all:.2%})')

[Diagnostic] Source test performance:
[Source (ToN-IoT) [post-pretrain]] F1: 0.9149 | Acc: 0.9355

[Diagnostic] Zero-shot on targets (no adaptation yet):
[Target1 (UNSW-NB15)  [zero-shot]] F1: 0.0491 | Acc: 0.5917
[Target2 (CICIDS2018) [zero-shot]] F1: 0.0063 | Acc: 0.8298

[Diagnostic] Parameters that will be updated during CTTA:
  Full CTTA:   20,610 / 103,810 params (19.85%)
  Spline-only: 18,432 / 103,810 params (17.76%)


## 8. Phase 2 — Zero-Shot Baseline

In [11]:
print('=' * 60)
print('SCALER EXPERIMENT: source scaler vs target scaler (zero-shot)')
print('=' * 60)

print('\n[Target scaler] (current default):')
evaluate(model, stream_tgt1, device, desc=f'Target1 ({TARGET1_NAME}) target-scaler')
evaluate(model, stream_tgt2, device, desc=f'Target2 ({TARGET2_NAME})  target-scaler')

print('\n[Source scaler] (same scaler as pre-training):')
evaluate(model, stream_tgt1_src_sc, device, desc=f'Target1 ({TARGET1_NAME}) source-scaler')
evaluate(model, stream_tgt2_src_sc, device, desc=f'Target2 ({TARGET2_NAME})  source-scaler')

SCALER EXPERIMENT: source scaler vs target scaler (zero-shot)

[Target scaler] (current default):
[Target1 (UNSW-NB15) target-scaler] F1: 0.0491 | Acc: 0.5917
[Target2 (CICIDS2018)  target-scaler] F1: 0.0063 | Acc: 0.8298

[Source scaler] (same scaler as pre-training):
[Target1 (UNSW-NB15) source-scaler] F1: 0.0986 | Acc: 0.5846
[Target2 (CICIDS2018)  source-scaler] F1: 0.2212 | Acc: 0.7079


0.2212120807697954

## 10. Ablation



In [12]:

# ============================================================================
# Standalone ablation block for paper Tables 3 and 4.
# Source = ToN-IoT, Target = UNSW-NB15, source scaler, KAN.
# Does not modify any earlier cell.
# ============================================================================
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score

model.load_state_dict(model_state)
# Define any hyperparameters the ablation needs but your notebook may not have
MAX_POOL_PASSES = globals().get('MAX_POOL_PASSES', 10**9)  # effectively no cap
TTA_LR          = globals().get('TTA_LR',          1e-2)
TTA_STEPS       = globals().get('TTA_STEPS',       1)
ENTROPY_W       = globals().get('ENTROPY_W',       1.0)
FEW_SHOT_W      = globals().get('FEW_SHOT_W',      1.0)
RECON_W_TTA     = globals().get('RECON_W_TTA',     0.5)

def _norm_params(model):
    """All LayerNorm / GroupNorm affine params anywhere in the model."""
    out = []
    for m in model.modules():
        if isinstance(m, (torch.nn.LayerNorm, torch.nn.GroupNorm)):
            out.extend(m.parameters())
    return out


def _classifier_params(model):
    return list(model.classifier.parameters())


def _last_layer_full_params(model):
    """ALL parameters of the last encoder layer (incl. base path for KAN)."""
    params = []
    if isinstance(model, KanAEClassifier):
        for p in model.encoder.layers[-1].parameters():
            params.append(p)
    elif isinstance(model, CnnAEClassifier):
        last = None
        for m in model.encoder.modules():
            if isinstance(m, (torch.nn.Conv1d, torch.nn.Linear)):
                last = m
        if last is not None:
            params.extend(last.parameters())
    elif isinstance(model, TabTransformerAEClassifier):
        params.extend(model.encoder_out.parameters())
    return params


def run_ctta_ablate(model, stream_loader, pool_loader, device,
                    trainable='spline_only',
                    use_ce=True, use_ent=True, use_recon=True):
    """Ablation-aware CTTA loop. Returns stream F1.

    trainable:
      'spline_only'      -> spline_weight + spline_scaler of last KAN layer
      'splines_norm'     -> splines + all LayerNorm / GroupNorm params
      'splines_cls'      -> splines + classifier head
      'last_layer_full'  -> ALL params of last encoder layer (incl. base path)
      'full'             -> norm + classifier + last encoder layer
    """
    for p in model.parameters():
        p.requires_grad = True

    if trainable == 'spline_only':
        train_params = get_spline_only_params(model)
    elif trainable == 'splines_norm':
        train_params = get_spline_only_params(model) + _norm_params(model)
    elif trainable == 'splines_cls':
        train_params = get_spline_only_params(model) + _classifier_params(model)
    elif trainable == 'splines_norm_cls':
        train_params = (get_spline_only_params(model)
                    + _norm_params(model)
                    + _classifier_params(model))
    elif trainable == 'last_layer_full':
        train_params = _last_layer_full_params(model)
    elif trainable == 'full':
        train_params = get_trainable_params(model)
    else:
        raise ValueError(f'unknown trainable={trainable!r}')

    trainable_ids = {id(p) for p in train_params}
    for p in model.parameters():
        p.requires_grad = (id(p) in trainable_ids)

    optimizer = torch.optim.Adam(train_params, lr=TTA_LR)
    ce_crit   = torch.nn.CrossEntropyLoss()
    model.train()

    pool_iter = iter(pool_loader)
    def next_pool():
        nonlocal pool_iter
        try:
            return next(pool_iter)
        except StopIteration:
            pool_iter = iter(pool_loader)
            return next(pool_iter)

    pool_size         = len(pool_loader.dataset)
    max_pool_samples  = MAX_POOL_PASSES * pool_size
    pool_samples_seen = 0

    all_preds, all_labels = [], []

    for x_stream, y_stream in stream_loader:
        x_stream = x_stream.to(device)

        for _ in range(TTA_STEPS):
            optimizer.zero_grad()
            loss = torch.tensor(0.0, device=device)

            if use_ent:
                logits_s, _, _ = model(x_stream)
                probs    = F.softmax(logits_s, dim=1)
                loss_ent = -torch.sum(probs * torch.log(probs + 1e-8), dim=1).mean()
                loss = loss + ENTROPY_W * loss_ent

            if (use_ce or use_recon) and pool_samples_seen < max_pool_samples:
                x_pool, y_pool = next_pool()
                x_pool = x_pool.to(device)
                y_pool = y_pool.to(device)
                logits_pool, recon_pool, _ = model(x_pool)

                if use_ce:
                    loss = loss + FEW_SHOT_W * ce_crit(logits_pool, y_pool)

                if use_recon:
                    benign = (y_pool == 0)
                    if benign.any():
                        loss = loss + RECON_W_TTA * F.mse_loss(
                            recon_pool[benign], x_pool[benign]
                        )

                pool_samples_seen += y_pool.size(0)

            if loss.requires_grad and not (torch.isnan(loss) or torch.isinf(loss)):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(train_params, max_norm=1.0)
                optimizer.step()

        with torch.no_grad():
            model.eval()
            preds = model(x_stream)[0].argmax(1).cpu().numpy()
            model.train()

        all_preds.extend(preds)
        all_labels.extend(y_stream.numpy())

    return f1_score(all_labels, all_preds, zero_division=0)


# ─── Table 3: trainable-subset ablation (5 rows) ───────────────────────────
print('=' * 60)
print('Table 3 — Trainable subset ablation')
print('=' * 60)
table3_order = [
    ('spline_only',     'Spline-only'),
    ('splines_norm',    'Splines + norm'),
    ('splines_cls',     'Splines + classifier'),
    ('splines_norm_cls', 'Splines + norm + classifier'),
    ('last_layer_full', 'Last KAN layer (incl. base path)'),
    ('full',            'Norm + classifier + last layer'),
]
t3 = {}
for key, label in table3_order:
    model.load_state_dict(model_state)
    t3[key] = run_ctta_ablate(
        model, stream_tgt2_src_sc, pool_tgt2_src_sc, device,
        trainable=key,
    )
    print(f'  {label:35s}  F1 = {t3[key]:.4f}')

# ─── Table 4: loss-term ablation (spline-only trainable set) ───────────────
print('\n' + '=' * 60)
print('Table 4 — Loss-term ablation (spline-only)')
print('=' * 60)
loss_configs = [
    ('CE + ent + recon',      True,  True,  True),
    ('CE + ent only',         True,  True,  False),
    ('CE only',               True,  False, False),
    ('ent only (TENT-like)',  False, True,  False),
]
t4 = {}
for name, ce, ent, recon in loss_configs:
    model.load_state_dict(model_state)
    t4[name] = run_ctta_ablate(
        model, stream_tgt2_src_sc, pool_tgt2_src_sc, device,
        trainable='spline_only',
        use_ce=ce, use_ent=ent, use_recon=recon,
    )
    print(f'  {name:25s}  F1 = {t4[name]:.4f}')

# ─── Compact summary ───────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('Paper values (rounded to 2 dp)')
print('=' * 60)
print('Table 3:')
for key, label in table3_order:
    print(f'  {label:35s} {t3[key]:.2f}')
print('Table 4:')
for name, *_ in loss_configs:
    print(f'  {name:25s} {t4[name]:.2f}')

Table 3 — Trainable subset ablation
  Spline-only                          F1 = 0.9310
  Splines + norm                       F1 = 0.9259
  Splines + classifier                 F1 = 0.9274
  Splines + norm + classifier          F1 = 0.9282
  Last KAN layer (incl. base path)     F1 = 0.9367
  Norm + classifier + last layer       F1 = 0.9325

Table 4 — Loss-term ablation (spline-only)
  CE + ent + recon           F1 = 0.9312
  CE + ent only              F1 = 0.9313
  CE only                    F1 = 0.9303
  ent only (TENT-like)       F1 = 0.2170

Paper values (rounded to 2 dp)
Table 3:
  Spline-only                         0.93
  Splines + norm                      0.93
  Splines + classifier                0.93
  Splines + norm + classifier         0.93
  Last KAN layer (incl. base path)    0.94
  Norm + classifier + last layer      0.93
Table 4:
  CE + ent + recon          0.93
  CE + ent only             0.93
  CE only                   0.93
  ent only (TENT-like)      0.22


In [13]:
print(f'Device={device} | Arch={ARCH} | [SOURCE]={SOURCE_NAME} | Size={SAMPLE_N} | Last Layer={SPLINE_TRUE}')
print(f"Last exec: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%SZ')}")

Device=cpu | Arch=kan | [SOURCE]=ToN-IoT | Size=1000000 | Last Layer=False
Last exec: 2026-05-26 03:02:33Z
